# Document length EDA (#3)

Compare document lengths in **SciFact, NFCorpus and FiQA** against the pilot model's input limits. Corpus-only analysis; no qrels or retrieval comparisons.

| Term | Meaning | Pilot model |
|---|---|---|
| Training length | Approximate training input length, not a hard limit | ~250 word pieces |
| `max_seq_length` | Inputs beyond this are truncated | 512 default |
| Chunking | Splits documents into multiple vectors rather than truncating | Not evaluated here |

**Counting:** Pilot WordPiece tokenizer, including `[CLS]`/`[SEP]`; `title + text` (FiQA has no titles). Report both 250 and 256 because the model card does not clarify special-token counting.

In [ ]:
import json
import platform
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentence_transformers
import transformers
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers.utils import logging as hf_logging

# we only count tokens with the raw tokenizer, so its "longer than 512" warning doesn't apply
hf_logging.set_verbosity_error()

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLES = ROOT / "results" / "tables"
FIGURES = ROOT / "results" / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MODEL = {
    "name": "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    "revision": "b207367332321f8e44f96e224ef15bc607f4dbf0",
}
DATASETS = {
    "scifact": {"repo": "BeIR/scifact", "revision": "b3b5335604bf5ee3c4447671af975ea25143d4f5"},
    "nfcorpus": {"repo": "BeIR/nfcorpus", "revision": "b5026a0e96e8a7ac4f95f482a596389289d46269"},
    "fiqa": {"repo": "BeIR/fiqa", "revision": "979c07a7cb5ccc6ca009792241fa1250b98055dd"},
}
THRESHOLDS = [128, 250, 256, 512]

VERSIONS = {
    "python": platform.python_version(),
    "transformers": transformers.__version__,
    "sentence-transformers": sentence_transformers.__version__,
    "datasets": datasets.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print(json.dumps(VERSIONS, indent=2))

In [ ]:
model = SentenceTransformer(MODEL["name"], revision=MODEL["revision"], device="cpu")
tokenizer = model.tokenizer
print("max_seq_length:", model.max_seq_length)
print("max position embeddings:", model[0].auto_model.config.max_position_embeddings)


def token_counts(texts):
    return np.array([len(ids) for ids in tokenizer(list(texts), add_special_tokens=True)["input_ids"]])

In [ ]:
docs = {}
for name, source in DATASETS.items():
    corpus = load_dataset(source["repo"], "corpus", split="corpus", revision=source["revision"]).to_pandas()
    title = corpus["title"].fillna("").str.strip()
    text = corpus["text"].fillna("").str.strip()
    doc_text = (title + " " + text).str.strip()
    docs[name] = pd.DataFrame(
        {
            "doc_id": corpus["_id"].astype(str),
            "has_title": title.ne(""),
            "empty": doc_text.eq(""),
            "tokens": token_counts(doc_text),
        }
    )
    print(f"{name}: {len(corpus)} docs")

## Distributions

Plots cap the x-axis at 1,024 tokens; longer documents fall in the last histogram bin. Dashed lines: 128 / 250 / 256 / 512.

In [ ]:
summary_rows, threshold_rows = [], []
for name, df in docs.items():
    t = df["tokens"]
    summary_rows.append(
        {
            "dataset": name,
            "n_docs": len(df),
            "pct_with_title": round(100 * df["has_title"].mean(), 1),
            "n_empty": int(df["empty"].sum()),
            "median": t.median(),
            "mean": round(t.mean(), 1),
            "p90": round(t.quantile(0.9), 1),
            "p99": round(t.quantile(0.99), 1),
            "max": t.max(),
        }
    )
    for k in THRESHOLDS:
        over = t > k
        threshold_rows.append(
            {
                "dataset": name,
                "threshold": k,
                "n_over": int(over.sum()),
                "pct_over": round(100 * over.mean(), 1),
                "pct_tokens_dropped": round(100 * (t - k).clip(lower=0).sum() / t.sum(), 1),
            }
        )

doc_summary = pd.DataFrame(summary_rows)
doc_thresholds = pd.DataFrame(threshold_rows)
doc_summary.to_csv(TABLES / "doc_length_summary.csv", index=False)
doc_thresholds.to_csv(TABLES / "doc_length_thresholds.csv", index=False)
doc_summary

In [ ]:
wide = doc_thresholds.pivot(index="dataset", columns="threshold", values=["pct_over", "pct_tokens_dropped"])
wide.loc[list(DATASETS)]

In [ ]:
doc_thresholds

## Distributions

The x-axis is cut at 1,024 tokens, and in the histograms every longer document lands in the last bin. The dashed lines mark 128 / 250 / 256 / 512.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, df in docs.items():
    t = np.sort(df["tokens"].to_numpy())
    ax.plot(t, np.arange(1, len(t) + 1) / len(t), label=f"{name} (n={len(t):,})")
for k in THRESHOLDS:
    ax.axvline(k, color="grey", linestyle="--", linewidth=0.8)
    ha = {250: "right", 256: "left"}.get(k, "center")
    ax.text(k, 1.01, f" {k} " if k in (250, 256) else str(k), ha=ha, va="bottom", fontsize=8, color="grey")
ax.set_xlim(0, 1024)
ax.set_ylim(0, 1)
ax.set_xlabel("Document length (tokens, incl. [CLS]/[SEP])")
ax.set_ylabel("Share of documents at or below length")
ax.set_title("Document length, all documents", pad=16)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES / "doc_length_ecdf.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(docs), figsize=(13, 3.5), sharey=False)
bins = np.arange(0, 1024 + 16, 16)
for ax, (name, df) in zip(axes, docs.items()):
    ax.hist(df["tokens"].clip(upper=1024), bins=bins)
    for k in THRESHOLDS:
        ax.axvline(k, color="grey", linestyle="--", linewidth=0.8)
    ax.set_title(f"{name} (n={len(df):,})")
    ax.set_xlabel("tokens (clipped at 1024)")
axes[0].set_ylabel("documents")
fig.tight_layout()
fig.savefig(FIGURES / "doc_length_hist.png", dpi=150)
plt.show()

In [ ]:
provenance = {"model": MODEL, "datasets": DATASETS, "thresholds": THRESHOLDS, "versions": VERSIONS, "platform": platform.platform()}
_ = (TABLES / "doc_length_provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")

## Summary

| | SciFact | NFCorpus | FiQA |
|---|---:|---:|---:|
| Documents | 5,183 | 3,633 | 57,638 |
| Median tokens | 316 | 357 | 120 |
| Over 250 / 256 | 73.2% / 71.0% | 80.3% / 78.8% | 19.9% / 19.2% |
| Over 512 | 8.8% | 9.1% | 4.2% |
| Tokens dropped at 256 / 512 | 28.9% / 3.1% | 32.2% / 2.6% | 20.2% / 5.8% |

- **SciFact / NFCorpus:** Most documents exceed ~250 tokens; fewer than 10% exceed 512.
- **FiQA:** Shorter documents overall, but a long tail; **38 empty documents**.
- **Limit:** Length alone does not show retrieval degradation. NFCorpus 256 vs. 512 retrieval results are in `nfcorpus_truncation.ipynb`; chunking/model selection is #5.